# Critical Input DEQN: Compare Rule-Based Policies

This notebook loads saved outputs from the fixed Taylor and bottleneck-adjusted Taylor notebooks and writes a compact comparison table.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
COMPARISON_DIR = ARTIFACT_ROOT / 'comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    'fixed_taylor': ARTIFACT_ROOT / 'fixed_taylor' / 'fixed_eval.json',
    'modified_taylor': ARTIFACT_ROOT / 'modified_taylor' / 'ba_eval.json',
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing {name} results: {path}')

In [ ]:
rows = []
for policy, path in paths.items():
    with path.open('r', encoding='utf-8') as fh:
        d = json.load(fh)
    row = {'policy': policy}
    for key in [
        'overall.rms', 'overall.max_abs', 'hh_euler.rms', 'labor.rms',
        'resource.rms', 'price_index.rms', 'calvo_S.rms', 'calvo_F.rms',
        'cap_fb.rms', 'repair_fb.rms', 'Q.rms',
    ]:
        row[key] = d.get(key)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('policy')
summary

In [ ]:
out_csv = COMPARISON_DIR / 'rule_eval_summary.csv'
summary.to_csv(out_csv)
out_csv